# SBN Generator - Genetic Algorithm Explorer

## Design Space: 2^6 = 64 Architectures

| C | Name | Definition |
|---|------|------------|
| **S** | Stratification | Alternating S/P layers |
| **A** | Acyclicity | Strict DAG (no cycles) |
| **R** | Regularity | Equal source→sink path lengths |
| **E** | Interleaving | Cross-block dependencies |
| **H** | Homogeneity | Identical functions per layer |
| **L** | Locality | Bounded connection distance |

**Circuit spec**: 16-bit input/output/state  
**Optimization**: Genetic Algorithm with configurable parameters  


## Step 0: Install Dependencies

In [ ]:
# Run if needed:
# !pip install pandas matplotlib
print("Dependencies: pandas, matplotlib")


## Step 1: Module Setup & CUDA Configuration

In [ ]:
import shutil, os, glob, sys

SRC = os.path.expanduser("~/Downloads/sbn_module_v4.py")
DST = os.path.expanduser("~/sbn_module_v4.py")

if os.path.exists(SRC):
    shutil.copy2(SRC, DST)
    print(f"Copied {SRC} -> {DST}")
elif os.path.exists(DST):
    print(f"Using existing {DST}")
else:
    print("WARNING: sbn_module_v4.py not found in ~/Downloads/")

for d in [os.path.expanduser("~/.cupy/kernel_cache"), "/tmp/cupy"]:
    if os.path.exists(d):
        for f in glob.glob(os.path.join(d, "**", "*.cubin*"), recursive=True):
            os.remove(f)

for mod in list(sys.modules):
    if "sbn_module" in mod:
        del sys.modules[mod]

cuda_lib = '/opt/conda/lib/python3.13/site-packages/nvidia/cuda_nvrtc/lib'
if os.path.exists(cuda_lib):
    if "LD_LIBRARY_PATH" in os.environ:
        os.environ["LD_LIBRARY_PATH"] = f"{cuda_lib}:{os.environ['LD_LIBRARY_PATH']}"
    else:
        os.environ["LD_LIBRARY_PATH"] = cuda_lib
    print("CUDA library path configured")

print("Ready.")


## Step 2: Imports

In [ ]:
import sys, numpy as np, random, time, csv
import matplotlib.pyplot as plt
from IPython.display import display, HTML, FileLink
import pandas as pd

from sbn_module_v4 import (
    create_sbn, mutate_sbn, GPUAccelerator, RewardFunctions,
    run_genetic_algorithm, constraints_to_label,
    all_64_architectures, CONSTRAINT_KEYS, CONSTRAINT_NAMES
)

np.random.seed(42)
random.seed(42)
print("Imports OK")


## Step 2b: GPU Memory Check & Cleanup

Run this if you get GPU memory errors.

In [ ]:
import subprocess
import gc

print("="*60)
print("GPU MEMORY CHECK")
print("="*60)

# Check nvidia-smi
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.free,memory.total', 
                           '--format=csv,noheader,nounits'], 
                          capture_output=True, text=True)
    if result.returncode == 0:
        used, free, total = map(int, result.stdout.strip().split(','))
        print(f"GPU Memory: {used} MB used, {free} MB free, {total} MB total")
        print(f"Usage: {used/total*100:.1f}%")
    else:
        print("Could not query GPU memory")
except Exception as e:
    print(f"Error checking GPU: {e}")

print()
print("="*60)
print("CLEARING MEMORY")
print("="*60)

# Force Python garbage collection
gc.collect()
print("✓ Python garbage collected")

# Clear CuPy cache if loaded
try:
    import cupy as cp
    mempool = cp.get_default_memory_pool()
    pinned = cp.get_default_pinned_memory_pool()
    mempool.free_all_blocks()
    pinned.free_all_blocks()
    print("✓ CuPy memory pools cleared")
except:
    print("CuPy not loaded yet")

print()
print("Run Step 3 to initialize GPU")


## Step 3: GPU Configuration

### 3a - GPU Initialization

In [ ]:
gpu = GPUAccelerator()
rewards = RewardFunctions(gpu)
print("GPU accelerator ready" if gpu.available else "CPU mode (GPU unavailable)")


### 3b - Truth Table Benchmark

In [ ]:
import time

print("="*60)
print("TRUTH TABLE BENCHMARK")
print("="*60)

sbn = create_sbn(S=True)

print("CPU truth table...")
t0 = time.time()
table_cpu = gpu._truth_table_cpu(sbn)
t_cpu = time.time() - t0
print(f"  {t_cpu:.3f}s")

if gpu.available:
    print("GPU truth table (cached kernel)...")
    t0 = time.time()
    table_gpu = gpu._truth_table(sbn)
    t_gpu = time.time() - t0
    print(f"  {t_gpu:.6f}s")
    print(f"  Speedup: {t_cpu/t_gpu:.0f}x")
else:
    print("GPU unavailable - CPU mode only")

print()
print("Note: Differential fitness uses CPU truth table + GPU DDT kernels")


### 3c - Differential Optimization Notes

**GPU Performance Achieved:**
- Truth table: ~0.04s (GPU)
- DDT computation: ~11.6s (GPU, batch sync optimized)
- **Total: ~12s per evaluation**

**Key optimizations applied:**
1. ✓ CUDA headers linked (`cuda_fp16.h` via symlinks in `/opt/conda/include`)
2. ✓ Truth table compiled on GPU (75× faster than CPU)
3. ✓ DDT kernel batch synchronization (sync every 100 iterations)
4. ✓ GPU max accumulation (single GPU→CPU transfer)

**Estimated times for 64 architectures:**
- pop=20, gen=50: **206 hours = 8.6 days** ✓
- pop=20, gen=40: **165 hours = 6.9 days**
- pop=30, gen=60: **372 hours = 15.5 days**

### 3d - Architecture Validation

Verify that all 64 architecture combinations generate valid SBN structures.

In [ ]:
def check_wellformed(sbn):
    if len(sbn.output_map) != 16:
        return False, f"Expected 16 outputs, got {len(sbn.output_map)}"
    all_ids = (set(sbn.nodes) |
               {f'input_{i}' for i in range(16)} |
               {f'state_{i}' for i in range(16)})
    for node in sbn.compute_graph:
        for inp in node.inputs:
            if inp not in all_ids:
                return False, f"Unknown input {inp} in {node.node_id}"
    return True, "OK"

archs   = all_64_architectures()
results = []

print(f"Validating {len(archs)} architectures...")
for arch in archs:
    label  = constraints_to_label(arch) or 'Generic'
    try:
        sbn = create_sbn(**arch)
        ok, msg = check_wellformed(sbn)
        if ok:
            sbn.reset([0]*16)
            sbn.step([1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0])
            status = "Valid"
        else:
            status = f'Error: {msg}'
    except Exception as e:
        status = f'Exception: {str(e)[:60]}'
    
    results.append({
        "Architecture": label,
        **{k: int(arch[k]) for k in CONSTRAINT_KEYS},
        "Status": status,
    })

df = pd.DataFrame(results)
display(df)

valid_count = sum(1 for r in results if r["Status"] == "Valid")
print(f"\nResult: {valid_count}/{len(archs)} architectures valid")


## Step 4: Fitness Functions

### 4a - Configuration

In [ ]:
# =========================================================================
# GA CONFIGURATION
# Edit these values, then run this cell
# =========================================================================

# --- Fitness function ---
# Options: "linear" | "algebraic" | "differential"
SELECTED_FITNESS = "differential"

# --- Architecture constraint ---
# Options: "S" | "-S" | "A" | "-A" | "R" | "-R" | "E" | "-E" | "H" | "-H" | "L" | "-L"
# - "X"  : Run GA for the 32 architectures where constraint X is active  (X = 1)
# - "-X" : Run GA for the 32 architectures where constraint X is inactive (X = 0)
SELECTED_CONSTRAINT = "S"

# --- GA Parameters ---
POPULATION_SIZE  = 20  # pop=20, gen=50 benchmarks: linear ~21 min, algebraic ~18 min, differential ~4.3 days
NUM_GENERATIONS  = 50
MUTATION_RATE    = 3.0

# --- Results storage ---
_ga_results = []

# =========================================================================
# DISPLAY CONFIGURATION SUMMARY
# =========================================================================

_constraint_map = dict(zip(CONSTRAINT_KEYS, CONSTRAINT_NAMES))

if SELECTED_CONSTRAINT.startswith("-"):
    _key  = SELECTED_CONSTRAINT[1:]
    _name = _constraint_map.get(_key, _key)
    arch_label = f"32 architectures where {_key} = 0  ({_name} inactive)"
    arch_count = 32
else:
    _key  = SELECTED_CONSTRAINT
    _name = _constraint_map.get(_key, _key)
    arch_label = f"32 architectures where {_key} = 1  ({_name} active)"
    arch_count = 32

fitness_labels = {
    "linear":       "Linear resistance (WHT)",
    "algebraic":    "Algebraic degree (ANF)",
    "differential": "Differential resistance (DDT, GPU optimized ~12s/eval)",
}
fit_label = fitness_labels.get(SELECTED_FITNESS, SELECTED_FITNESS)

display(HTML(
    "<div style='font-family:monospace;font-size:13px;background:#f9f9f9;"
    "border:1px solid #ccc;border-radius:6px;padding:16px;max-width:600px;'>"
    "<div style='font-weight:bold;font-size:15px;margin-bottom:12px;"
    "border-bottom:1px solid #ccc;padding-bottom:6px;'>GA Configuration</div>"

    f"<table style='width:100%;border-collapse:collapse;margin-bottom:12px'>"
    f"<tr><td style='padding:4px 0;width:160px'><b>Fitness:</b></td>"
    f"<td style='padding:4px 0'>{SELECTED_FITNESS} — {fit_label}</td></tr>"
    f"<tr><td style='padding:4px 0'><b>Architecture:</b></td>"
    f"<td style='padding:4px 0'>{arch_label}</td></tr>"
    f"<tr><td style='padding:4px 0'><b>Runs:</b></td>"
    f"<td style='padding:4px 0'>{arch_count}</td></tr>"
    f"<tr><td style='padding:4px 0'><b>Population:</b></td><td style='padding:4px 0'>{POPULATION_SIZE}</td></tr>"
    f"<tr><td style='padding:4px 0'><b>Generations:</b></td><td style='padding:4px 0'>{NUM_GENERATIONS}</td></tr>"
    f"<tr><td style='padding:4px 0'><b>Mutation rate:</b></td><td style='padding:4px 0'>{MUTATION_RATE}</td></tr>"
    f"</table>"

    f"<div style='padding:8px 12px;background:#e3f2fd;border-left:3px solid #2196f3;font-size:11px'>"
    f"Run <b>Step 4b</b> below to execute the GA with these parameters."
    f"</div>"

    "</div>"
))


# =========================================================================
# TIME ESTIMATION
# =========================================================================

# Realistic number of fitness evaluations:
# GA evaluates approximately the full population each generation
# (initial pop + mutations, minus small elitism benefit)
total_evals = arch_count * POPULATION_SIZE * NUM_GENERATIONS

# Measured times per eval (benchmarks: pop=20, gen=50, 32 archs)
# linear: ~21 min total  -> 0.039 s/eval
# algebraic: ~18 min total -> 0.034 s/eval
# differential: ~12 s/eval (CPU truth table + GPU DDT)
time_per_eval = {
    "linear":       0.039,
    "algebraic":    0.034,
    "differential": 12.0,
}

est_seconds = total_evals * time_per_eval.get(SELECTED_FITNESS, 1.0)
est_minutes = est_seconds / 60
est_hours   = est_seconds / 3600

print()
print("="*60)
print("ESTIMATED EXECUTION TIME")
print("="*60)
print(f"Total evaluations: {total_evals:,}")
print(f"Time per eval: ~{time_per_eval.get(SELECTED_FITNESS, 1.0):.2f}s")
print()
if est_hours < 1:
    print(f"Estimated time: {est_minutes:.1f} minutes")
else:
    print(f"Estimated time: {est_hours:.1f} hours ({est_minutes:.0f} minutes)")
    if est_hours > 24:
        print(f"               ({est_hours/24:.1f} days)")
print("="*60)


### 4b - Run Genetic Algorithm

In [ ]:
# =========================================================================
# EXECUTE GA
# =========================================================================
# IMPORTANT: Edit RUN_GA below to control execution
# Set RUN_GA = True when ready to start

RUN_GA = False  # ← Change to True to run, False to skip

# =========================================================================

def _resolve_architectures(constraint):
    """Return (archs, label, count) matching SELECTED_CONSTRAINT.

    - "X"  -> 32 architectures where constraint X is active  (X = 1)
    - "-X" -> 32 architectures where constraint X is inactive (X = 0)
    """
    all_archs = all_64_architectures()
    if constraint.startswith("-"):
        key   = constraint[1:]
        archs = [a for a in all_archs if not a[key]]
        return archs, f"32 architectures where {key} = 0", 32
    else:
        key   = constraint
        archs = [a for a in all_archs if a[key]]
        return archs, f"32 architectures where {key} = 1", 32

_archs, _arch_label, _num_runs = _resolve_architectures(SELECTED_CONSTRAINT)

if not RUN_GA:
    time_per_eval = {
        "linear":       0.039,
        "algebraic":    0.034,
        "differential": 12.0,
    }

    total_evals = _num_runs * POPULATION_SIZE * NUM_GENERATIONS
    est_seconds = total_evals * time_per_eval.get(SELECTED_FITNESS, 1.0)
    est_minutes = est_seconds / 60
    est_hours   = est_seconds / 3600

    print("="*70)
    print("GA NOT STARTED (RUN_GA = False)")
    print("="*70)
    print()
    print("Configuration:")
    print(f"  Fitness:      {SELECTED_FITNESS}")
    print(f"  Architecture: {_arch_label}")
    print(f"  Runs:         {_num_runs}")
    print(f"  Population:   {POPULATION_SIZE}")
    print(f"  Generations:  {NUM_GENERATIONS}")
    print()
    print("Estimated execution time:")
    print(f"  Total evaluations: {total_evals:,}")
    print(f"  Time per eval: ~{time_per_eval.get(SELECTED_FITNESS, 1.0):.1f}s")
    if est_hours < 1:
        print(f"  Total time: {est_minutes:.1f} minutes")
    else:
        print(f"  Total time: {est_hours:.1f} hours ({est_minutes:.0f} min)")
        if est_hours > 24:
            print(f"             ({est_hours/24:.1f} days)")
    print()
    print("="*70)
    print("To run the GA:")
    print("  1. Review configuration in Step 4a")
    print("  2. Set RUN_GA = True (line 6)")
    print("  3. Re-run this cell")
    print("="*70)
    print()
else:
    # Map fitness to functions
    _fitness_map = {
        "linear":       (gpu.linear_resistance,      True),
        "algebraic":    (gpu.algebraic_degree,        True),
        "differential": (gpu.differential_resistance, False),
    }

    if SELECTED_FITNESS not in _fitness_map:
        raise ValueError(f"Unknown fitness: {SELECTED_FITNESS}")

    fitness_fn, maximize = _fitness_map[SELECTED_FITNESS]

    print("="*70)
    print("STARTING GA")
    print("="*70)
    print(f"Fitness:      {SELECTED_FITNESS}")
    print(f"Architecture: {_arch_label}")
    print(f"Runs:         {_num_runs}")
    print(f"Population:   {POPULATION_SIZE}")
    print(f"Generations:  {NUM_GENERATIONS}")
    print("="*70)
    print()

    results = []
    t_start  = time.time()

    for i, arch in enumerate(_archs, 1):
        label = constraints_to_label(arch) or 'Generic'
        print(f"\n[{i}/{_num_runs}] {label}")

        def progress(gen, total_gen, best):
            if gen % 10 == 0 or gen == total_gen - 1:
                print(f"  Gen {gen+1}/{total_gen}: best={best:.6f}", flush=True)

        t0 = time.time()
        best_sbn, best_score, history = run_genetic_algorithm(
            constraints=arch,
            fitness_fn=fitness_fn,
            maximize=maximize,
            population_size=POPULATION_SIZE,
            num_generations=NUM_GENERATIONS,
            mutation_rate=MUTATION_RATE,
            progress_callback=progress,
        )
        elapsed = time.time() - t0

        print(f"  → Final: {best_score:.6f}  ({elapsed:.1f}s)")

        results.append({
            "Architecture": label,
            **{k: int(arch[k]) for k in CONSTRAINT_KEYS},
            "Best_Score": round(float(best_score), 6),
            "Time_s":     round(elapsed, 1),
        })

    total_time = time.time() - t_start

    print()
    print("="*70)
    print("COMPLETED")
    print("="*70)

    df = (pd.DataFrame(results)
            .sort_values("Best_Score", ascending=not maximize)
            .reset_index(drop=True))
    df.index = range(1, _num_runs + 1)
    display(df)

    print(f"\nTotal time: {total_time/60:.1f} min ({total_time/3600:.1f} hours)")

    fname = f'ga_results_{SELECTED_FITNESS}_{SELECTED_CONSTRAINT}_pop{POPULATION_SIZE}_gen{NUM_GENERATIONS}.csv'
    df.to_csv(fname, index_label="Rank")
    display(FileLink(fname, result_html_prefix="Download CSV: "))

    _ga_results.extend(results)


## Step 5: Results Visualization

Visualize GA results from Step 4b.

In [ ]:

if not _ga_results:
    print("No results yet. Run Step 4b first (with RUN_GA = True).")
else:
    df = pd.DataFrame(_ga_results)
    
    print("="*60)
    print(f"SUMMARY: {len(df)} GA run(s) completed")
    print("="*60)
    display(df)
    print()
    
    # Group by architecture if multiple runs
    if "Architecture" in df.columns and len(df) > 1:
        print("="*60)
        print("BEST SCORE BY ARCHITECTURE")
        print("="*60)
        
        grouped = df.groupby("Architecture")["Best_Score"].agg(["min", "max", "mean", "count"])
        grouped = grouped.sort_values("mean")
        display(grouped)
        print()
        
        # Plot if we have multiple architectures
        if len(grouped) > 1:
            fig, ax = plt.subplots(figsize=(10, 6))
            archs = grouped.index.tolist()
            means = grouped["mean"].tolist()
            
            ax.barh(archs, means, color='#2962ff', alpha=0.7)
            ax.set_xlabel("Mean Best Score", fontsize=12)
            ax.set_ylabel("Architecture", fontsize=12)
            ax.set_title("GA Performance by Architecture", fontsize=14)
            ax.grid(True, axis='x', alpha=0.3)
            plt.tight_layout()
            plt.show()
    
    # Runtime analysis
    if "Time_s" in df.columns:
        print("="*60)
        print("RUNTIME ANALYSIS")
        print("="*60)
        print(f"Total time: {df['Time_s'].sum()/60:.1f} minutes")
        print(f"Mean time per run: {df['Time_s'].mean():.1f}s")
        print(f"Min/Max: {df['Time_s'].min():.1f}s / {df['Time_s'].max():.1f}s")
